# 3.6 Project: Docs Q&A Bot

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Learning objectives

- Assemble a complete RAG system from a specification: ingestion, chunking, embeddings, retrieval, generation, evaluation.
- Defend each pipeline choice with evidence from your own eval numbers, not with tutorial defaults.
- Build an eval harness first, and use it to make at least one real design decision.
- Report cost and latency honestly, with budgets your bot provably stays under.

## Prerequisites

All of Module 3: document loading and chunking (3.1), embeddings and vector stores (3.2), retrieval techniques (3.3), grounded generation and citations (3.4), and RAG evaluation (3.5). You will reuse the golden-set format, the RAGAS scoring loop, and the hit-rate script.

## The brief

Build a docs Q&A bot: a program a user runs in the terminal, points at a set of documents, and asks questions about. Pick a real document set yourself: a product's documentation, an open-source project's docs, a policy handbook, a corpus of markdown notes. The set must be big enough that retrieval is a real problem: at least 50 pages or the equivalent in markdown. Five pages is a demo; fifty pages is a retrieval system.

A realistic session looks like this:

> "What is the refund policy for annual plans?"
> "The refund policy for annual plans is 30 days from purchase [2]. Monthly plans have a 14-day window [5]."

And like this, when the docs have no answer:

> "Does the tool support on-premise deployment?"
> "I could not find anything about on-premise deployment in the documentation I was given."

This is the second thing in the course you build without step-by-step instructions. The specification is the whole assignment. Reading a spec and turning it into a working system, with the numbers to prove it works, is the skill this course is actually teaching.

## Required components

Design the system yourself, but it must include these five parts. Suggested approaches are in brackets; use them or your own, and defend the choice in your README.

1. **Ingest and chunk** (lesson 3.1). Your documents must become chunks with stable ids (doc name plus index is fine). Defend the chunking choice in writing: size, overlap, and why this corpus suits it. Run the retrieval hit-rate check from lesson 3.5 against at least two chunking configs and keep the winner, with the numbers in your README.
2. **Embeddings and vector store** (3.2). An embedding model and a vector store that runs locally (FAISS and Chroma both qualify). State the model and its dimensions. Rebuilding the index must be a single command.
3. **Retrieval** (3.3). Not plain top-k similarity alone: MMR, hybrid search, or similarity plus a reranker. Defend the choice with hit-rate numbers against the baseline, not with a feeling.
4. **Grounded answers with numbered citations** (3.4). Every factual claim in every answer carries a citation like `[2]` that maps to a retrieved chunk, and the bot abstains cleanly when the context has no answer. The citation format is a hard requirement, not a styling choice: if claims cannot be traced to chunks, the bot has failed.
5. **An eval harness with results** (3.5). A golden set of at least 10 questions in the lesson-3.5 format (question, reference answer, expected chunk ids), including at least two abstention probes and one multi-hop question. Report measured faithfulness and context recall means, plus the retrieval hit rate at k. The README includes the numbers; numbers that are not in the README do not exist.

Also required, because production bots live or die on these: measure and report **cost and latency per query** (embeddings + retrieval + generation, timed end to end), set explicit budgets (for example, under 2 seconds p50 and under five cents per query), and show that the bot stays under them on your golden set.

## Non-negotiable requirements

These come straight from the module, and the rubric scores them:

- **Citations on every factual claim.** No claim without a `[n]` pointing at a retrieved chunk. If the answer contains a sentence you cannot cite, rewrite the sentence or cut it.
- **Abstention when context is missing.** "I could not find this in the docs" beats a fluent guess every time. The golden set's abstention probes must pass: the bot must not answer from parametric memory.
- **Eval results in the README.** Faithfulness, context recall, hit rate, cost, latency. Measured, not estimated.
- **A real document set, not a toy.** At least 50 pages equivalent. If you pick docs you wrote yourself, say so; the eval questions must still be written before you see the retrieval results.
- **Reproducibility.** `requirements.txt` with pinned versions (ragas==0.4.3 included), a single command to build the index, a single command to run the eval. Someone else clones the repo and gets your numbers.

## Suggested structure

You own the architecture, but a shape that works well:

In [ ]:
docs_qa/
  ingest.py        # load docs -> chunk -> stable ids -> vector store
  retrieve.py      # retriever config: k, MMR/hybrid, reranker
  answer.py        # grounded generation: prompt, citations, abstention rule
  eval_qa.py       # golden set runner: hit rate, RAGAS scores, cost/latency
  cli.py           # terminal loop: ask, answer, show citations
  eval/golden.json # the golden set
  README.md        # choices defended with numbers, budgets, results

Keep the generation prompt short and behavioural: answer only from context, cite every claim, say plainly when the context is missing the answer. Put the citation discipline in the prompt and the citation check in code if you can: a post-answer scan that flags any factual sentence without a citation marker is cheap to write and catches the exact failure mode this module cares about.

## Stretch goals (pick at most two)

- **Streaming CLI.** Stream the answer token by token instead of printing it whole. Teaches you the streaming interface you will need in Module 4 and Module 6.
- **Local-embeddings variant.** Swap the hosted embedding model for a local one (via langchain-ollama or sentence-transformers) and report the quality and cost delta. This is the data-residency story: an EU customer whose documents must never leave their servers gets the same bot with no API calls at all. Write down exactly where quality drops.
- **Incremental re-index.** Handle documents being added or changed without rebuilding the whole index: track file hashes, re-embed only what changed. Real docs change; a bot that needs a full rebuild per edit is a prototype.
- **Citation verification pass.** A second small model pass that checks each citation marker against the chunk it points to, and drops or corrects claims whose citations do not check out. Teaches you what layered reliability costs in latency.

## Acceptance checklist

Run through this list yourself before you consider the project done. Every item must pass.

- [ ] Index builds with one command over at least 50 pages; chunks carry stable ids.
- [ ] Ask a question answered on one page; the answer is correct and every factual claim has a numbered citation.
- [ ] Ask a question needing facts from two pages; both pages are cited.
- [ ] Ask a question the docs cannot answer; the bot abstains instead of guessing (check that it did not answer from parametric memory).
- [ ] Ask with a synonym the docs do not use ("cost" vs "pricing"); the answer is still found.
- [ ] Golden set of 10+ questions exists, with at least 2 abstention probes and 1 multi-hop question, in the lesson-3.5 format.
- [ ] `eval_qa.py` reports faithfulness, context recall, and hit rate at k; numbers are in the README.
- [ ] At least one design decision (chunking config or retrieval method) was changed because of eval numbers, and the README says which and by how much.
- [ ] Cost and latency per query are measured on the golden set and under the stated budgets.
- [ ] A second run of the eval after a pipeline change shows the numbers moved in the expected direction (or you can explain why they did not).

## Rubric (how this is assessed)

| Criterion | Weight | What excellence looks like |
|---|---|---|
| Retrieval quality | 25% | Hit rate and context recall measured and good; retrieval choice defended against a baseline with numbers; multi-hop and synonym probes pass. |
| Grounded answers | 25% | Every claim cited; citations map to the right chunks; abstention works and is tested; no parametric-memory leakage on probes. |
| Evaluation discipline | 25% | Golden set of 10+ with the required probe types; faithfulness and recall reported; at least one decision driven by eval numbers; README carries all results. |
| Engineering | 15% | One-command index build and eval; pinned requirements; stable chunk ids; cost/latency budgets set and met; code is readable. |
| Honesty under failure | 10% | README documents what does not work yet and where quality drops (e.g. the local-embeddings variant); failure modes from lesson 3.5 addressed, not hidden. |

Score yourself honestly against this before the module assessment. If a number is weak, that is a finding: write down why you think it is weak and what you would try next.

## What can go wrong

- **The golden set is written after seeing the answers.** Questions shaped around what the bot already answers well produce a perfect score and zero information. Write the set from the docs, before the first retrieval run.
- **Citations that point at the wrong chunk.** A `[2]` after a true sentence is still a failure if chunk 2 does not say it. Check citations against chunks, not against truth.
- **Abstention that never triggers.** If every abstention probe gets an answer, your abstention instruction is decoration. Test it deliberately; it is the highest-value probe in the set.
- **Chunk ids that change between runs.** Without stable ids, expected_chunk_ids in the golden set rots the first time you re-chunk, and hit rate becomes meaningless. Doc name plus index, assigned once.
- **Eval-driven overfitting.** Tuning chunking until the 10-question set scores perfectly is the same disease as training on the test set. Keep a few held-out questions you never tune against, and say so in the README.
- **Latency nobody measured.** Retrieval plus rerank plus generation plus a verification pass adds up; measure end to end on the golden set, not per component in isolation.
- **Documents with personal data.** If your document set is real customer docs, support tickets, or anything with names and emails, the GDPR note from lesson 3.5 applies to the index and the logs too. A local vector store keeps the data on your machine; a hosted one needs the same data-processing agreement as any other service.

## Review questions

**1. Why does the brief require the chunking decision to be defended with hit-rate numbers instead of just stated?**
Because chunking is the highest-leverage choice in the pipeline and the one most often copied from tutorials without thought. Two configs on your corpus, one number each, is the cheapest real experiment in the module, and it teaches the habit the whole course depends on: decisions backed by your own measurements, not by defaults.

**2. Your bot answers every question correctly but half the citations point at chunks that do not contain the cited fact. Which rubric criterion fails, and what is the fix?**
Grounded answers fails: citations are the traceability mechanism, and wrong citations are worse than none because they look trustworthy. The fix is citation verification: check each marker against its chunk's text (the stretch goal), or tighten the generation prompt so claims stay close to chunk wording, then re-run the golden set.

**3. You tune the chunk size until faithfulness on your 10-question set hits 1.0. What is wrong with this, and what does the brief require instead?**
That is tuning on the test set: the number now measures how well you fit ten questions, not how good the pipeline is. The brief requires held-out questions you never tune against, and at least one design decision driven by numbers is expected, but the numbers must come from a set the tuning did not see.

**4. A learner's README reports great faithfulness but no cost or latency numbers. What is missing, and why does it matter?**
The engineering criterion is incomplete. A bot that answers well at 12 seconds and 40 cents per query is a different product from one at 1.5 seconds and 2 cents. Budgets force the trade-offs (smaller judge, no reranker, local embeddings) to be made consciously instead of discovered in production.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.